# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [2]:
import pandas as pd
import plotly.express as px
import numpy as np

# The quiz uses the World Population Dataset (Kaggle).
# Try a public mirror of the same dataset first. If unavailable, upload
# world_population.csv in the notebook's Files panel and rerun this cell.
DATA_URL = "https://raw.githubusercontent.com/amaantayyab/World-Population/main/world_population.csv"

try:
    df = pd.read_csv("world_population.csv")
except FileNotFoundError:
    try:
        df = pd.read_csv(DATA_URL)
    except Exception as e:
        from google.colab import files
        print("Automatic download failed. Please upload world_population.csv.")
        uploaded = files.upload()
        df = pd.read_csv(next(iter(uploaded)))

# Standardize column names used in the quiz instructions
df = df.rename(columns={
    'Country/Territory': 'Country',
    'Area (km²)': 'Area',
    'Density (per km²)': 'Density',
    'World Population Percentage': 'World share (%)'
})

year_cols = [c for c in df.columns if c.endswith('Population') and c[:4].isdigit()]
df_long = df.melt(
    id_vars=['CCA3', 'Country', 'Continent'],
    value_vars=year_cols,
    var_name='Year',
    value_name='Population'
)
df_long['Year'] = df_long['Year'].str[:4].astype(int)

# Common formatting helpers
PALETTE = {
    'context': '#B8BEC7',
    'accent': '#2878B5',
    'positive': '#2878B5',
    'negative': '#D18B35'
}
print(f"Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Tidy data: {df_long.shape[0]:,} rows × {df_long.shape[1]} columns")
display(df.head())


Dataset: 234 rows × 16 columns
Tidy data: 1,872 rows × 5 columns


,Rank,CCA3,Country,Capital,Continent,2022 Population,2020 Population,2015 Population,2010 Population,2000 Population,1990 Population,1980 Population,1970 Population,Area,Density,World share (%)
0,36,AFG,Afghanistan,Kabul,Asia,41128771,38972230,33753499,28189672,19542982,10694796,12486631,10752971,652230,63.0587,0.52
1,138,ALB,Albania,Tirana,Europe,2842321,2866849,2882481,2913399,3182021,3295066,2941651,2324731,28748,98.8702,0.04
2,34,DZA,Algeria,Algiers,Africa,44903225,43451666,39543154,35856344,30774621,25518074,18739378,13795915,2381741,18.8531,0.56
3,213,ASM,American Samoa,Pago Pago,Oceania,44273,46189,51368,54849,58230,47818,32886,27075,199,222.4774,0.00
4,203,AND,Andorra,Andorra la Vella,Europe,79824,77700,71746,71519,66097,53569,35611,19860,468,170.5641,0.00


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** I use a **line chart** because the question asks how each continent changed over time. I index each continent to its own 1970 population (= 100), so the slopes compare *relative growth* rather than absolute population size. A single blue accent highlights the fastest-growing continent; the other lines provide context.

In [3]:
# Q1 — Relative population growth since 1970 (1970 = 100)
years = sorted(df_long['Year'].unique())
base = (df_long[df_long['Year'] == 1970]
        .set_index('CCA3')['Population'])
continent_year = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
base_continent = continent_year[continent_year['Year'] == 1970].set_index('Continent')['Population']
continent_year['Index (1970=100)'] = continent_year.apply(
    lambda r: 100 * r['Population'] / base_continent.loc[r['Continent']], axis=1
)
latest = continent_year[continent_year['Year'] == max(years)].sort_values('Index (1970=100)', ascending=False)
fastest = latest.iloc[0]['Continent']
fig = px.line(continent_year, x='Year', y='Index (1970=100)', color='Continent',
              title=f"{fastest} has grown fastest since 1970",
              color_discrete_sequence=px.colors.qualitative.Safe,
              markers=True, template='plotly_white')
fig.update_layout(xaxis_title=None, yaxis_title='Population index (1970 = 100)',
                  legend_title=None, hovermode='x unified')
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()
print("Fastest-growing continent:", fastest)
display(latest[['Continent', 'Index (1970=100)']].round(1))


Fastest-growing continent: Africa


,Continent,Index (1970=100)
7,Africa,390.4
39,Oceania,231.2
47,South America,226.4
15,Asia,220.1
31,North America,190.3
23,Europe,113.3


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** A **sorted horizontal bar chart** makes the 15 largest absolute increases easy to rank and compare. Bar length encodes additional people, while a single accent highlights African countries among the top 15; direct labels make the scale readable.

In [4]:
# Q2 — 15 countries with the largest absolute population increase
df['Population added (1970–2022)'] = df['2022 Population'] - df['1970 Population']
top15 = df.nlargest(15, 'Population added (1970–2022)').sort_values('Population added (1970–2022)')
top15['Region'] = np.where(top15['Continent'].eq('Africa'), 'Africa', 'Other continents')
africa_count = int(top15['Continent'].eq('Africa').sum())
fig = px.bar(top15, x='Population added (1970–2022)', y='Country', orientation='h',
             color='Region', color_discrete_map={'Africa': PALETTE['accent'], 'Other continents': PALETTE['context']},
             text='Population added (1970–2022)',
             title=f"15 countries added the most people; {africa_count} are in Africa",
             template='plotly_white')
fig.update_traces(texttemplate='%{text:.2s}', textposition='outside', cliponaxis=False)
fig.update_layout(xaxis_title='Additional people, 1970–2022', yaxis_title=None,
                  legend_title=None, showlegend=True)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()
print(f"Number of African countries in the top 15: {africa_count}")
display(top15[['Country','Continent','Population added (1970–2022)']].sort_values(
    'Population added (1970–2022)', ascending=False))


Number of African countries in the top 15: 4


,Country,Continent,Population added (1970–2022)
92,India,Asia,859671872
41,China,Asia,603352887
156,Pakistan,Asia,176533990
149,Nigeria,Africa,162971948
93,Indonesia,Asia,160272945
221,United States,North America,137961517
27,Brazil,South America,118943623
16,Bangladesh,Asia,103644512
63,Ethiopia,Africa,95071678
55,DR Congo,Africa,78858479


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** A **bubble scatter plot** shows the relationship between land area and population while colour separates continents and bubble size adds population context. Logarithmic axes make countries spanning several orders of magnitude comparable. Hover labels identify outliers without cluttering the figure.

In [5]:
# Q3 — Land area and 2022 population
fig = px.scatter(df, x='Area', y='2022 Population', color='Continent',
                 size='World share (%)', size_max=38, hover_name='Country',
                 hover_data={'Area':':,.0f','2022 Population':':,.0f','World share (%)':':.2f'},
                 log_x=True, log_y=True,
                 title='Land area and population are related only loosely',
                 template='plotly_white', opacity=0.78,
                 color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(xaxis_title='Land area (km², log scale)',
                  yaxis_title='Population in 2022 (log scale)', legend_title=None)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

# Identify notable departures from the overall area–population pattern
corr = df[['Area','2022 Population']].corr(method='spearman').iloc[0,1]
print(f"Spearman rank correlation between area and population: {corr:.2f}")
print("Explore the labelled hover points for countries with unusually large populations for their area,")
print("or very large land areas relative to their populations.")


Spearman rank correlation between area and population: 0.85
Explore the labelled hover points for countries with unusually large populations for their area,
or very large land areas relative to their populations.


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** A **violin plot** shows the full distribution of country-level population density within each continent, including skew and spread. A logarithmic y-axis prevents a few very dense countries from flattening the rest of the distributions.

In [6]:
# Q4 — Distribution of population density by continent
fig = px.violin(df, x='Continent', y='Density', color='Continent',
                box=True, points='outliers', log_y=True,
                title='Population density varies widely within every continent',
                template='plotly_white',
                color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(xaxis_title=None, yaxis_title='People per km² (log scale)',
                  showlegend=False)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()
display(df.groupby('Continent')['Density'].agg(['count','median','mean','min','max']).round(1))


,count,median,mean,min,max
Continent,,,,,
Africa,57,59.7,125.0,2.2,871.9
Asia,50,123.0,1025.0,2.2,23172.3
Europe,50,106.9,663.3,3.6,18234.5
North America,40,221.4,272.8,0.0,1299.3
Oceania,23,78.2,132.5,3.4,603.2
South America,14,17.8,21.0,0.3,65.0


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** A **choropleth map** is appropriate because the question asks where growth and decline occurred geographically. Country colour encodes percentage change from 2000 to 2022 using a diverging, non-red/green palette centred on zero.

In [7]:
# Q5 — Geographic pattern of population percentage change, 2000–2022
df['Population change (%)'] = (
    (df['2022 Population'] - df['2000 Population']) / df['2000 Population'] * 100
)
fig = px.choropleth(
    df, locations='CCA3', locationmode='ISO-3', color='Population change (%)',
    hover_name='Country',
    hover_data={'CCA3':False, 'Population change (%)':':.1f',
                '2000 Population':':,.0f', '2022 Population':':,.0f'},
    color_continuous_scale=['#D18B35', '#F2F2F2', '#2878B5'],
    color_continuous_midpoint=0,
    title='Population growth dominates, but some countries have declined',
    template='plotly_white'
)
fig.update_layout(geo=dict(showframe=False, showcoastlines=True),
                  coloraxis_colorbar_title='Change (%)')
fig.show()
display(df[['Country','Continent','Population change (%)']].sort_values(
    'Population change (%)').head(10))


,Country,Continent,Population change (%)
153,Northern Mariana Islands,Oceania,-38.321840
3,American Samoa,Oceania,-23.968745
117,Lithuania,Europe,-23.601880
126,Marshall Islands,Oceania,-23.338374
133,Moldova,Europe,-23.016822
111,Latvia,Europe,-22.648786
25,Bosnia and Herzegovina,Europe,-22.630888
226,Vatican City,Europe,-21.658986
229,Wallis and Futuna,Oceania,-21.401888
218,Ukraine,Europe,-18.776723


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** A **treemap** displays the hierarchy in one view: continent first, then the countries within each continent. Rectangle area is proportional to 2022 population, making both the continental split and the largest country contributions visible.

In [8]:
# Q6 — 2022 population by continent and country
fig = px.treemap(
    df, path=['Continent','Country'], values='2022 Population',
    color='Continent', hover_name='Country',
    hover_data={'2022 Population':':,.0f'},
    title='Asia contains the largest share of the world’s population',
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig.update_traces(textinfo='label+percent parent', textfont_size=12)
fig.update_layout(margin=dict(t=60,l=10,r=10,b=10), showlegend=False)
fig.show()
continent_totals = df.groupby('Continent')['2022 Population'].sum().sort_values(ascending=False)
display((continent_totals / continent_totals.sum() * 100).rename('World share (%)').round(1))


,World share (%)
Continent,
Asia,59.2
Africa,17.9
Europe,9.3
North America,7.5
South America,5.5
Oceania,0.6


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** A **100% stacked bar chart** compares each continent's share of the global population across years. Each bar sums to 100%, so changes in share are directly comparable even as the total world population grows.

In [9]:
# Q7 — Population share over time; the remaining continents grouped together
focus = ['Africa', 'Asia', 'Europe']
shares = df_long.copy()
shares['Continent group'] = np.where(shares['Continent'].isin(focus),
                                     shares['Continent'], 'Rest of the world')
continent_year = shares.groupby(['Year','Continent group'], as_index=False)['Population'].sum()
world_year = continent_year.groupby('Year', as_index=False)['Population'].sum().rename(
    columns={'Population':'World population'})
continent_year = continent_year.merge(world_year, on='Year')
continent_year['World share (%)'] = (
    continent_year['Population'] / continent_year['World population'] * 100
)
order = ['Africa','Asia','Europe','Rest of the world']
fig = px.bar(continent_year, x='Year', y='World share (%)',
             color='Continent group', barmode='stack',
             category_orders={'Continent group':order},
             color_discrete_map={'Africa':'#2878B5','Asia':'#D18B35',
                                 'Europe':'#7A8793','Rest of the world':'#D9DEE5'},
             title='Africa’s share is rising while Europe’s share is falling',
             template='plotly_white')
fig.update_layout(xaxis_title=None, yaxis_title='Share of world population (%)',
                  legend_title=None)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, range=[0,100])
fig.show()
display(continent_year[continent_year['Year'].isin([1970,2022])]
        .pivot(index='Continent group', columns='Year', values='World share (%)')
        .reindex(order).round(1))


Year,1970,2022
Continent group,,
Africa,9.9,17.9
Asia,58.1,59.2
Europe,17.8,9.3
Rest of the world,14.3,13.6


---

## Submission note

The analysis and Plotly figures are implemented above. The notebook expects the supplied `world_population.csv` file. If it is not already in the runtime, the setup cell first tries a public mirror; if that is unavailable, upload the CSV from the Kaggle dataset linked in the introduction and rerun the setup cell. Run the notebook from top to bottom so all seven interactive figures and result tables are generated.